# Docker 入門

**学習目標**
  * Docker コンテナのビルドと実行
  * Docker Hub および Google Artifact Registry からの Docker イメージの pull
  * Google Artifact Registry への Docker イメージの push

## 概要

Docker は、アプリケーションの開発、配信、実行を行うためのオープン プラットフォームです。Docker を使用すると、アプリケーションをインフラストラクチャから分離し、インフラストラクチャをマネージド アプリケーションのように扱うことができます。Docker を活用することで、コードの迅速な配信、テスト、デプロイが可能になり、コードの作成から実行までのサイクルを短縮できます。

Docker は、カーネルのコンテナ化機能と、アプリケーションの管理やデプロイを支援するワークフローおよびツールを組み合わせることでこれを実現しています。

Docker コンテナは Kubernetes で直接利用できるため、Kubernetes Engine 上で簡単に実行できます。Docker の基本を学ぶことで、Kubernetes やコンテナ化されたアプリケーションの開発を始めるためのスキルを習得できます。

## 基本的な Docker コマンド

現在存在する Docker イメージを確認します。

In [ ]:
!docker images

Docker を初めて使用する場合は、リポジトリが一覧に表示されません。

`docker run` を使用して、パブリック レジストリから `hello-world` という Docker イメージを pull してみましょう。Docker デーモンは `hello-world` イメージを検索し、ローカルに見つからない場合は Docker Hub というパブリック レジストリからイメージを pull して、そのイメージからコンテナを作成し、実行します。

In [ ]:
!docker run hello-world

これで Docker イメージを確認すると、`hello-world` も表示されるはずです。

In [ ]:
!docker images

これは Docker Hub パブリック レジストリから pull されたイメージです。Image ID は `SHA256` ハッシュ形式になっており、プロビジョニングされた Docker イメージを特定します。Docker デーモンはローカルにイメージが見つからない場合、デフォルトでパブリック レジストリからイメージを検索します。コンテナをもう一度実行してみましょう。

これで、再度 `docker run hello-world` を実行する場合、Artifact Registry からダウンロードする必要はありません。

実行中のすべての Docker コンテナを確認するには、`docker ps` を使用します。

In [ ]:
!docker ps

実行中のコンテナはありません。**注: AI Notebook 上で実行している場合は、1 つの実行中コンテナが表示されます。**

先ほど実行した `hello-world` コンテナはすでに終了しています。実行が完了したコンテナも含め、すべてのコンテナを確認するには、`docker ps -a` を実行します。

In [ ]:
!docker ps -a

ここには、コンテナを識別するために Docker によって生成された UUID である Container ID や、実行に関する詳細なメタデータが表示されます。コンテナの Name もランダムに生成されますが、`docker run --name [container-name] hello-world` を使用して指定することもできます。

## Docker コンテナのビルド

シンプルな Node アプリケーションをベースにした Docker イメージをビルドしてみましょう。

**演習**

`dockerfiles` フォルダにある `intro.docker` というテキストファイルを開き、そこにある TODO を完了してください。

Dockerfile には次のステップを含める必要があります。

 1. `FROM` を使用して、親イメージとして公式の Node ランタイム（例: `node:6`）を継承する
 2. `WORKDIR` を使用して、作業ディレクトリを `/app` に設定する
 3. `ADD` を使用して、カレント ディレクトリの内容をコンテナの `/app` にコピーする
 4. `EXPOSE` を使用して、コンテナのポート 80 を外部に公開する
 5. `CMD` を使用して、コマンド `node ./src/app.js` を実行する

このファイルは、イメージのビルド方法を Docker デーモンに指示します。

最初の行ではベースとなる親イメージを指定します。ここでは Node バージョン 6 の公式 Docker イメージです。
2 行目では、コンテナの作業（カレント）ディレクトリを設定します。
3 行目では、カレント ディレクトリの内容（「.」で指定）をコンテナに追加します。
次に、そのポートで接続を受け入れられるようにコンテナのポートを公開し、最後に Node コマンドを実行してアプリケーションを起動します。

各行の役割を理解するために、その他の [Docker コマンド リファレンス](https://docs.docker.com/engine/reference/builder/#known-issues-run) も確認してください。

この Docker コンテナを使用してシンプルな Node.js アプリを実行します。`app.js` を見てみましょう。これはポート 80 でリッスンし、「Hello World」を返すシンプルな HTTP サーバーです。


それではイメージをビルドしましょう。「`.`」はカレント ディレクトリを意味するため、Dockerfile があるディレクトリ内からこのコマンドを実行する必要がある点に改めて注意してください。

`-t` は、`name:tag` の構文でイメージに名前とタグを付けるためのものです。イメージ名は `node-app`、タグは `0.1` です。Docker イメージをビルドする際は、タグを付けることを強くお勧めします。タグを指定しない場合、デフォルトで `latest` に設定されるため、新しいイメージと古いイメージの区別が難しくなります。また、イメージのビルドに伴い、上記の Dockerfile の各行が中間コンテナ レイヤとして生成されることにも注目してください。

**演習**

`docker build` を使用して `dockerfiles/intro.docker` の Docker イメージをビルドしてください。イメージには `node-app:0.1` というタグを付けます。

In [ ]:
%%bash
TODO: Your code goes here

イメージが正しく作成されたか確認してみましょう。

In [ ]:
!docker images

数秒前に作成された `node-app` リポジトリが表示されているはずです。

`node` がベースイメージであり、`node-app` がビルドしたイメージであることに注目してください。`node-app` を先に削除しないと、`node` を削除することはできません。イメージのサイズは VM と比較して比較的小さくなっています。`node:slim` や `node:alpine` など、他のバージョンの Node イメージを使用すると、さらに小さいイメージを作成して移植性を高めることができます。コンテナ サイズの軽量化については、高度なトピックでさらに詳しく解説されています。すべてのバージョンは公式リポジトリで確認できます。

なお、Docker イメージからイメージを削除するには、`docker rmi [repository]:[tag]` を使用します。

## Docker コンテナの実行

次に、`docker run` コマンドを使用して、上記でビルドしたイメージに基づいてコンテナを実行します。`--name` フラグを使用すると、必要に応じてコンテナに名前を付けることができます。また、`-p` はホストのポート 4000 をコンテナのポート 80 にマッピングするよう Docker に指示します。これにより、http://localhost:4000 でサーバーにアクセスできるようになります。ポート マッピングを行わない場合、localhost からコンテナにアクセスすることはできません。

In [ ]:
!docker ps -a

**演習**

`docker run` を使用して、先ほどビルドした `node-app:0.1` というコンテナを実行してください。ホスト ポート `4000` をポート `80` に割り当て、名前を `my-app` に設定します。

In [ ]:
%%bash
TODO: Your code goes here

サーバーをテストするには、ターミナル ウィンドウを開いて次のコマンドを入力します。

```bash
curl http://localhost:4000
```

サーバーから `Hello World` という応答が返されるはずです。

コンテナは、起動したターミナルが実行されている間、動作し続けます。コンテナを停止したい場合は、ターミナルで次のコマンドを実行してコンテナを停止および削除します。

```bash
docker stop my-app && docker rm my-app
```
しばらくするとコンテナが停止します。上記のセルの実行が完了したことが確認できるはずです。

#### バックグラウンドでのコンテナの実行
コンテナをバックグラウンドで（ターミナルのセッションに紐付けずに）実行したい場合は、`-d` フラグを指定する必要があります。
次のコマンドを実行して、コンテナをバックグラウンドで起動してみましょう。

**演習**

上記のコマンドに `-d` フラグを追加して、`my-app` をバックグラウンドで実行するように修正してください。

In [ ]:
%%bash
TODO: Your code goes here

コンテナがバックグラウンドで実行されるようになりました。`docker ps` を使用して、実行中のコンテナのステータスを確認できます。

In [ ]:
!docker ps

`docker ps` の出力でコンテナが実行中であることを確認してください。`docker logs [container_id]` を実行することでログを確認できます。

In [ ]:
# 注: コンテナ ID は環境によって異なります
!docker logs b9d5fd6b8e33

次のように表示されるはずです。
```bash
Server running at http://0.0.0.0:80/
```
コンテナの実行中にログ出力をリアルタイムで追跡したい場合は、`-f` オプションを使用します。

## 変更と公開

アプリケーションを変更し、Google Cloud Artifact Registry に push してみましょう。Artifact Registry リポジトリは、リポジトリのセットアップ プロセスですでに `asl-artifact-repo` として作成されています（詳細は **[セットアップ コード](../../../scripts/setup_on_jupyterlab.sh)** を参照）。

その後、クリーンな環境を再現するためにすべてのローカル コンテナとイメージを削除し、Artifact Registry からコンテナを pull して実行します。これにより、Docker コンテナの移植性が実証されます。

### `app.js` の編集
テキスト エディタで `./src/app.js` ファイルを開き、"Hello World" を別の文字列に置き換えます。その後、この新しいイメージをビルドします。

**演習**

`app.js` ファイルを変更した後、同じ Dockerfile から `docker build` を使用して `node-app:0.2` という名前の新しいコンテナをビルドしてください。

In [ ]:
%%bash
TODO: Your code goes here

出力の `Step 2` で既存のキャッシュ レイヤが使用されていることに注目してください。`Step 3` 以降は、`app.js` に変更を加えたためレイヤが変更されています。

新しいイメージ バージョンで別のコンテナを実行します。ホストのポート 80 ではなく 8000 をマッピングしていることに注目してください。ホスト ポート 4000 は既に使用されているため使用できません。

**演習**

別のポートを使用し、`my-app-2` という名前で、この新しいコンテナをバックグラウンドで実行してください。

In [ ]:
%%bash
TODO: Your code goes here

`docker ps` を使用して、両方のコンテナが実行中であることを確認できます。

In [ ]:
!docker ps

前回と同様に、`curl` を使用して両方のコンテナをテストしてみましょう。

In [ ]:
!curl http://localhost:8000

In [ ]:
!curl http://localhost:4000

実行中のコンテナを停止するには、ターミナルで実行するか、（バックグラウンドで実行されているため）このノートブックのセルで次のコマンドを実行します。

### Artifact Registry への公開

次に、イメージを Google Cloud Artifact Registry に push します。Artifact Registry でホストされているプライベート レジストリにイメージを push するには、イメージにレジストリ名を含むタグを付ける必要があります。形式は `[hostname]/[project-id]/[repository name]/[image]:[tag]` です。

Artifact Registry の場合:

  * `[hostname]`= `<location>-docker.pkg.dev`。ここでは `us-docker-pkg.dev` を使用します。
  * `[project-id]`= ご自身のプロジェクト ID
  * `[repository name]`= リポジトリ名。ここでは `asl-artifact-repo` を使用します。
  * `[image]`= イメージ名
  * `[tag]`= 任意の文字列タグ。指定しない場合、デフォルトは `latest` になります。

In [ ]:
import os

PROJECT_ID = "your-gcp-project-here"  # ご自身のプロジェクト名に置き換えてください

os.environ["PROJECT_ID"] = PROJECT_ID

`node-app:0.2` にタグを付けましょう。

In [ ]:
!docker images

**演習**
`node-app:0.2` イメージに、命名規則 `us-docker.pkg.dev/[project-id]/[repository name]/[image]:[tag]` に準拠した新しいイメージ名でタグを付けてください。イメージ名とタグ名は同じままにします。

In [ ]:
%%bash
TODO: Your code goes here

Docker イメージを一覧表示すると、この新しくタグ付けされたリポジトリが表示されるはずです。

In [ ]:
!docker images

イメージを Artifact Registry に push する前に、docker コマンドを認証する必要があります。

In [ ]:
%%bash
gcloud auth print-access-token | docker login -u oauth2accesstoken --password-stdin https://us-docker.pkg.dev

次に、このイメージを Artifact Registry に push しましょう。

**演習**

この新しいイメージを Artifact Registry に push してください。

In [ ]:
%%bash
TODO: Your code goes here

Cloud Console のイメージ レジストリにアクセスして、イメージが Artifact Registry に存在することを確認します。コンソールで `ナビゲーション メニュー > Artifact Registry` に移動するか、下のセルの URL にアクセスします。

In [ ]:
%%bash
echo "http://us-docker.pkg.dev/${PROJECT_ID}/asl-artifact-repo/node-app"

### 公開された Artifact Registry イメージのテスト

このイメージをテストしてみましょう。新しい VM を起動し、その VM に SSH 接続して gcloud をインストールすることもできます。ここでは簡単にするために、クリーンな環境を再現する目的ですべてのコンテナとイメージを削除します。

まず、`docker stop` と `docker rm` を使用して、すべてのコンテナを停止および削除します。

In [ ]:
!docker stop my-app && docker rm my-app

In [ ]:
!docker stop my-app-2 && docker rm my-app-2

次に、`docker rmi` を使用して、上記で作成した Docker イメージを削除します。

In [ ]:
!docker images

In [ ]:
%%bash
docker rmi node-app:0.2
docker rmi us-docker.pkg.dev/${PROJECT_ID}/asl-artifact-repo/node-app:0.2
docker rmi node-app:0.1
docker rmi node:6 
docker rmi -f hello-world:latest

`docker images` で、すべてのイメージが削除されたことを確認します。

In [ ]:
!docker images

この時点で、擬似的なクリーン環境が整いました。それでは、イメージを pull して実行してみましょう。

In [ ]:
%%bash
docker pull us-docker.pkg.dev/${PROJECT_ID}/asl-artifact-repo/node-app:0.2
docker run -p 4000:80 -d us-docker.pkg.dev/${PROJECT_ID}/asl-artifact-repo/node-app:0.2

先ほどと同様に、期待どおりに動作していることを確認できます。

In [ ]:
!curl http://localhost:4000

Copyright 2020 Google LLC Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at https://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.